# 🔎 08 — Single Image Inference (No SAHI)

Interactive notebook to inspect detections using **direct Ultralytics inference**
(no SAHI slicing) on any test image.

| Model | Box Color | Weights |
|-------|-----------|----------|
| YOLO26m | 🔵 Blue | `runs/detect/.../best.pt` |
| Custom | 🟠 Orange | `custom.pt` |

Set `IMAGE_NAME` or leave blank for random. **Re-run the cell** for a new image.

In [ ]:
# ============================================================
#  Setup — Run ONCE
# ============================================================

import random
from pathlib import Path

import torch
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from ultralytics import YOLO

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

PROJECT_ROOT    = Path("..")
TEST_IMG_DIR    = PROJECT_ROOT / "Dataset" / "test" / "images"
TEST_LBL_DIR    = PROJECT_ROOT / "Dataset" / "test" / "labels"
YOLO26M_WEIGHTS = PROJECT_ROOT / "runs" / "detect" / "retail_shelf_yolo26m" / "weights" / "best.pt"
CUSTOM_WEIGHTS  = PROJECT_ROOT / "custom.pt"

IMGSZ = 640
CONF  = 0.25
IOU   = 0.5
DEVICE = 0 if torch.cuda.is_available() else "cpu"

TEST_IMAGES = sorted(
    list(TEST_IMG_DIR.glob("*.jpg")) +
    list(TEST_IMG_DIR.glob("*.jpeg")) +
    list(TEST_IMG_DIR.glob("*.png"))
)


def load_gt(lbl_path, w, h):
    boxes = []
    if not lbl_path.exists(): return boxes
    with open(lbl_path) as f:
        for line in f:
            p = line.strip().split()
            if len(p) >= 5:
                cx, cy, bw, bh = map(float, p[1:5])
                boxes.append([(cx-bw/2)*w, (cy-bh/2)*h, (cx+bw/2)*w, (cy+bh/2)*h])
    return boxes


def predict_direct(model, img_path):
    results = model.predict(source=str(img_path), imgsz=IMGSZ, conf=CONF,
                            iou=IOU, device=DEVICE, verbose=False)
    preds = []
    for r in results:
        for box in r.boxes:
            preds.append({
                "bbox": box.xyxy[0].cpu().numpy().tolist(),
                "score": float(box.conf[0]),
            })
    return preds


def show_gt_vs_pred(model, img_path, model_name, box_color):
    preds = predict_direct(model, img_path)
    img = Image.open(img_path)
    w, h = img.size
    gt = load_gt(TEST_LBL_DIR / f"{img_path.stem}.txt", w, h)

    img_gt = img.copy(); d = ImageDraw.Draw(img_gt)
    for x1, y1, x2, y2 in gt:
        d.rectangle([x1, y1, x2, y2], outline="#00E676", width=4)

    img_pr = img.copy(); d = ImageDraw.Draw(img_pr)
    for p in preds:
        x1, y1, x2, y2 = p["bbox"]
        d.rectangle([x1, y1, x2, y2], outline=box_color, width=4)
        d.text((x1, max(0, y1-16)), f"{p['score']:.2f}", fill=box_color)

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(28, 14))
    a1.imshow(img_gt)
    a1.set_title(f"Ground Truth — {len(gt)} objects", fontsize=14, fontweight="bold", color="#00C853")
    a1.axis("off")
    a2.imshow(img_pr)
    a2.set_title(f"{model_name} — {len(preds)} detections", fontsize=14, fontweight="bold", color=box_color)
    a2.axis("off")
    plt.suptitle(f"{model_name} (No SAHI) — {img_path.name}\nGT: {len(gt)}  |  Preds: {len(preds)}",
                 fontsize=16, fontweight="bold", y=1.02)
    plt.tight_layout(); plt.show()
    print(f"📊 {model_name}  |  GT: {len(gt)}  |  Preds: {len(preds)}")


print(f"Device: {DEVICE}  |  Test images: {len(TEST_IMAGES)}")
print("✅ Setup complete.")

In [ ]:
# ============================================================
#  Load both models (run ONCE)
# ============================================================

print("Loading YOLO26m...")
yolo26m_model = YOLO(str(YOLO26M_WEIGHTS.resolve()))
print("✅ YOLO26m loaded.\n")

print("Loading Custom model...")
custom_model = YOLO(str(CUSTOM_WEIGHTS.resolve()))
print("✅ Custom model loaded.")

---
## 🔵 YOLO26m — GT vs Prediction (No SAHI)

In [ ]:
IMAGE_NAME = ""  # <-- e.g. "abc123.jpg", or blank for random

if IMAGE_NAME.strip():
    _p = TEST_IMG_DIR / IMAGE_NAME
    assert _p.exists(), f"Not found: {_p}"
else:
    _p = random.choice(TEST_IMAGES)
    print(f"🎲 Random pick: {_p.name}\n")

show_gt_vs_pred(yolo26m_model, _p, "YOLO26m", "#2196F3")

---
## 🟠 Custom Model — GT vs Prediction (No SAHI)

In [ ]:
IMAGE_NAME = ""  # <-- e.g. "abc123.jpg", or blank for random

if IMAGE_NAME.strip():
    _p = TEST_IMG_DIR / IMAGE_NAME
    assert _p.exists(), f"Not found: {_p}"
else:
    _p = random.choice(TEST_IMAGES)
    print(f"🎲 Random pick: {_p.name}\n")

show_gt_vs_pred(custom_model, _p, "Custom Model", "#FF6B35")

---
## ⚔️ Both Models — 3-Panel Comparison (No SAHI)

In [ ]:
IMAGE_NAME = ""  # <-- e.g. "abc123.jpg", or blank for random

if IMAGE_NAME.strip():
    _p = TEST_IMG_DIR / IMAGE_NAME
    assert _p.exists(), f"Not found: {_p}"
else:
    _p = random.choice(TEST_IMAGES)
    print(f"🎲 Random pick: {_p.name}\n")

# Predict with both
preds_y = predict_direct(yolo26m_model, _p)
preds_c = predict_direct(custom_model, _p)

img = Image.open(_p)
w, h = img.size
gt = load_gt(TEST_LBL_DIR / f"{_p.stem}.txt", w, h)

# Panel 1: GT
ig = img.copy(); d = ImageDraw.Draw(ig)
for x1, y1, x2, y2 in gt:
    d.rectangle([x1, y1, x2, y2], outline="#00E676", width=4)

# Panel 2: YOLO26m
iy = img.copy(); d = ImageDraw.Draw(iy)
for p in preds_y:
    x1, y1, x2, y2 = p["bbox"]
    d.rectangle([x1, y1, x2, y2], outline="#2196F3", width=4)
    d.text((x1, max(0, y1-16)), f"{p['score']:.2f}", fill="#2196F3")

# Panel 3: Custom
ic = img.copy(); d = ImageDraw.Draw(ic)
for p in preds_c:
    x1, y1, x2, y2 = p["bbox"]
    d.rectangle([x1, y1, x2, y2], outline="#FF6B35", width=4)
    d.text((x1, max(0, y1-16)), f"{p['score']:.2f}", fill="#FF6B35")

fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(36, 14))
a1.imshow(ig); a1.set_title(f"Ground Truth\n{len(gt)} objects", fontsize=14, fontweight="bold", color="#00C853"); a1.axis("off")
a2.imshow(iy); a2.set_title(f"YOLO26m\n{len(preds_y)} detections", fontsize=14, fontweight="bold", color="#2196F3"); a2.axis("off")
a3.imshow(ic); a3.set_title(f"Custom\n{len(preds_c)} detections", fontsize=14, fontweight="bold", color="#FF6B35"); a3.axis("off")

plt.suptitle(f"No SAHI — {_p.name}\nGT: {len(gt)}  |  YOLO26m: {len(preds_y)}  |  Custom: {len(preds_c)}",
             fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout(); plt.show()
print(f"📊 {_p.name}  |  GT: {len(gt)}  |  YOLO26m: {len(preds_y)}  |  Custom: {len(preds_c)}")